# Optimized LoRA — `twitter-roberta-base-sentiment` for Amazon reviews

## 1. Setup

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
os.environ["HF_HUB_DISABLE_XET"] = "1"
DATA_DIR = "/Users/nico/Downloads/archive"
FILES = ["1429_1.csv",
         "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv",
         "Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv"]
LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = 3
SEED = 42
MODEL_ID = "cardiffnlp/twitter-roberta-base-sentiment-latest"
device = "cuda" if torch.cuda.is_available() else ("mps" if getattr(torch.backends, "mps", None)
          and torch.backends.mps.is_available() else "cpu")
print("device:", device)

## 2. Load & clean

In [ ]:
df = pd.concat([pd.read_csv(f"{DATA_DIR}/{f}", low_memory=False)[["reviews.text", "reviews.rating"]]
                for f in FILES], ignore_index=True)
df.columns = ["text", "rating"]
df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df = df[(df["text"].str.len() >= 10) & df["rating"].between(1, 5)]
df = df.drop_duplicates(subset=["text", "rating"]).reset_index(drop=True)
df["sentiment"] = pd.cut(df["rating"], [0, 2, 3, 5], labels=LABELS)
df["label"] = df["sentiment"].cat.codes
print(f"{len(df):,} reviews after cleaning")
df["sentiment"].value_counts().reindex(LABELS)

## 3. Exploratory graphs

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df["rating"].value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#5b8def", title="Star ratings")
ax[0].set_xlabel("stars"); ax[0].set_ylabel("count")
df["sentiment"].value_counts().reindex(LABELS).plot(kind="bar", ax=ax[1],
        color=["#c0392b", "#e8912a", "#2b7bba"], title="Sentiment classes")
ax[1].set_xlabel(""); ax[1].set_ylabel("count")
plt.tight_layout(); plt.show()
imb = df["sentiment"].value_counts()
print(f"imbalance (positive : negative) = {imb['positive'] / imb['negative']:.1f} : 1")

In [ ]:
df["n_words"] = df["text"].str.split().str.len()
plt.figure(figsize=(9, 3.5))
for lab, c in zip(LABELS, ["#c0392b", "#e8912a", "#2b7bba"]):
    sns.kdeplot(df.loc[df.sentiment == lab, "n_words"].clip(upper=120), label=lab, color=c, fill=True, alpha=.15)
plt.xlabel("words per review (clipped at 120)"); plt.title("Review length by sentiment"); plt.legend()
plt.tight_layout(); plt.show()

## 4. Split & class weights

In [ ]:
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=.20, random_state=SEED, stratify=df["sentiment"])
train_df, val_df = train_test_split(train_df, test_size=.10, random_state=SEED, stratify=train_df["sentiment"])
print(f"train: {len(train_df):,}   val: {len(val_df):,}   test: {len(test_df):,}")
WEIGHT_POWER = 1.0
counts = train_df["label"].value_counts().sort_index().values
inv = counts.sum() / (NUM_LABELS * counts)
class_weights = torch.tensor(inv ** WEIGHT_POWER, dtype=torch.float)
class_weights = class_weights / class_weights.mean()
print("class weights:", {LABELS[i]: round(float(w), 2) for i, w in enumerate(class_weights)})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].bar(LABELS, counts, color=["#c0392b", "#e8912a", "#2b7bba"]); ax[0].set_title("Train counts"); ax[0].set_yscale("log")
ax[1].bar(LABELS, class_weights.numpy(), color=["#c0392b", "#e8912a", "#2b7bba"]); ax[1].set_title("Loss weight per class")
plt.tight_layout(); plt.show()

## 5. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
def to_ds(frame):
    ds = Dataset.from_pandas(frame[["text", "label"]].reset_index(drop=True).rename(columns={"label": "labels"}))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)
train_ds, val_ds, test_ds = to_ds(train_df), to_ds(val_df), to_ds(test_df)

## 6. LoRA model

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True)
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.1, bias="none", task_type="SEQ_CLS",
    target_modules=["query", "key", "value", "output.dense", "intermediate.dense"],
    modules_to_save=["classifier"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 7. Weighted-loss training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import f1_score, accuracy_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1_macro": f1_score(labels, preds, average="macro"),
            "accuracy": accuracy_score(labels, preds)}
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits.view(-1, NUM_LABELS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss
_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir="artifacts/lora_optimized",
    learning_rate=2e-4, num_train_epochs=4, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, weight_decay=0.01, warmup_ratio=0.06,
    eval_strategy="epoch", save_strategy="epoch", logging_steps=50,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    fp16=torch.cuda.is_available() and not _bf16, bf16=_bf16, report_to="none",
)
trainer = WeightedTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
    class_weights=class_weights, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()
trainer.save_model("artifacts/lora_optimized/best"); tokenizer.save_pretrained("artifacts/lora_optimized/best")

## 8. Training curves

In [ ]:
hist = pd.DataFrame(trainer.state.log_history)
tr = hist[hist.get("loss").notna()] if "loss" in hist else pd.DataFrame()
ev = hist[hist.get("eval_loss").notna()] if "eval_loss" in hist else pd.DataFrame()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
if len(tr):
    ax[0].plot(tr["step"], tr["loss"], color="#5b8def", label="train loss")
if len(ev):
    ax[0].plot(ev["step"], ev["eval_loss"], color="#c0392b", marker="o", label="val loss")
ax[0].set_xlabel("step"); ax[0].set_title("Loss"); ax[0].legend()
if len(ev):
    ax[1].plot(ev["epoch"], ev["eval_f1_macro"], color="#2b7bba", marker="o")
    ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].set_title("Validation macro-F1")
plt.tight_layout(); plt.show()

## 9. Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(test_ds)
y_true, y_pred = pred.label_ids, pred.predictions.argmax(-1)
lora_f1 = f1_score(y_true, y_pred, average="macro")
print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))
print(f"macro-F1: {lora_f1:.3f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(4.6, 3.8))
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues",
            xticklabels=LABELS, yticklabels=LABELS)
plt.xlabel("predicted"); plt.ylabel("actual"); plt.title("Confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1, 2])
x = np.arange(3); w = .25
plt.figure(figsize=(8, 3.4))
plt.bar(x - w, p, w, label="precision", color="#5b8def")
plt.bar(x,     r, w, label="recall",    color="#e8912a")
plt.bar(x + w, f, w, label="F1",        color="#2b7bba")
plt.xticks(x, LABELS); plt.ylim(0, 1); plt.title("Per-class metrics (optimized LoRA)"); plt.legend()
for i in range(3): plt.text(i + w, f[i] + .02, f"{f[i]:.2f}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

## 10. Error analysis — where the errors come from

In [ ]:
import re
_neu = test_df[test_df.sentiment == "neutral"].copy()
_pos = r"\b(love|great|excellent|perfect|awesome|amazing|best|wonderful|fantastic|happy|recommend|easy|good|nice)\b"
_neg = r"\b(hate|terrible|awful|worst|broken|useless|disappointed|disappointing|waste|poor|bad|refund|returned|slow|won't|doesn't work)\b"
_neu["pos_hits"] = _neu.text.str.lower().str.count(_pos)
_neu["neg_hits"] = _neu.text.str.lower().str.count(_neg)
_neu["reads"] = np.select(
    [_neu.pos_hits > _neu.neg_hits, _neu.neg_hits > _neu.pos_hits],
    ["reads_positive", "reads_negative"], default="mixed/flat")
print("How the 3-star (neutral) test reviews actually read:")
print((_neu["reads"].value_counts(normalize=True) * 100).round(1).astype(str) + " %")
print("\nExamples of 3-star reviews with positive language (labelled neutral):")
for t in _neu.loc[_neu.reads == "reads_positive", "text"].head(4):
    print("  \u2022", t[:120])
_vc = _neu["reads"].value_counts().reindex(["reads_negative", "mixed/flat", "reads_positive"])
plt.figure(figsize=(6, 3))
bars = plt.bar(_vc.index, _vc.values, color=["#c0392b", "#e8912a", "#2b7bba"])
plt.bar_label(bars, labels=[f"{v/_vc.sum()*100:.0f}%" for v in _vc.values], padding=2, fontsize=9)
plt.ylabel("reviews"); plt.title("How 3-star (neutral) reviews actually read")
plt.tight_layout(); plt.show()

## 11. Post-hoc decision tuning (the fix)

In [ ]:
import torch as _torch
test_probs = _torch.softmax(_torch.tensor(pred.predictions), dim=1).numpy()
val_out    = trainer.predict(val_ds)
val_probs  = _torch.softmax(_torch.tensor(val_out.predictions), dim=1).numpy()
val_true   = val_out.label_ids
train_priors = np.bincount(train_df["label"], minlength=NUM_LABELS) / len(train_df)
print("train priors:", {LABELS[i]: round(float(p), 3) for i, p in enumerate(train_priors)})

In [ ]:
def _macro(y, p): return f1_score(y, p, average="macro")
def fit_decision(vp, vy, priors):
    """Search prior-correction and class-multiplier rules; keep the best on val macro-F1."""
    cands = [("argmax", lambda P: P.argmax(1))]
    for tau in np.linspace(0.0, 1.5, 16):
        cands.append((f"prior^{tau:.2f}",
                      (lambda t: (lambda P: (P / (priors ** t + 1e-12)).argmax(1)))(tau)))
    for wneg in np.linspace(1.0, 4.0, 7):
        for wneu in np.linspace(1.0, 6.0, 11):
            w = np.array([wneg, wneu, 1.0])
            cands.append((f"mult={w.round(2).tolist()}",
                          (lambda ww: (lambda P: (P * ww).argmax(1)))(w)))
    scored = sorted([(n, _macro(vy, fn(vp)), fn) for n, fn in cands],
                    key=lambda x: x[1], reverse=True)
    return scored[0]
best_name, best_val_f1, decide = fit_decision(val_probs, val_true, train_priors)
print(f"val macro-F1:  argmax = {_macro(val_true, val_probs.argmax(1)):.3f}"
      f"  ->  '{best_name}' = {best_val_f1:.3f}")

In [ ]:
y_pred_tuned = decide(test_probs)
lora_f1_tuned = f1_score(y_true, y_pred_tuned, average="macro")
print(classification_report(y_true, y_pred_tuned, target_names=LABELS, digits=3))
print(f"macro-F1  argmax -> tuned:  {lora_f1:.3f}  ->  {lora_f1_tuned:.3f}"
      f"   ({lora_f1_tuned - lora_f1:+.3f})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.8))
for a, yp, title in [(ax[0], y_pred, f"argmax  (F1={lora_f1:.3f})"),
                     (ax[1], y_pred_tuned, f"tuned  (F1={lora_f1_tuned:.3f})")]:
    cm = confusion_matrix(y_true, yp)
    sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues",
                xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=a)
    a.set_xlabel("predicted"); a.set_ylabel("actual"); a.set_title(title)
plt.tight_layout(); plt.show()

## 10. Compare against every model

In [ ]:
results = {
    "LoRA twitter-roberta (baseline)":  0.522,
    "cardiffnlp (zero-shot)":           0.577,
    "roberta-base (full fine-tune)":    0.653,
    "LoRA twitter-roberta (optimized)": float(lora_f1),
    "LoRA twitter-roberta (+ tuned decision)": float(lora_f1_tuned),
}
summary = (pd.DataFrame({"model": list(results), "macro_F1": list(results.values())})
             .sort_values("macro_F1", ascending=False).reset_index(drop=True))
print(summary.to_string(index=False))
colors = ["#2b7bba" if ("optimized" in m or "tuned" in m) else "#9bb3c9" for m in summary["model"]]
ax = summary.set_index("model")["macro_F1"].plot(kind="barh", color=colors, figsize=(8, 3.2))
ax.invert_yaxis(); ax.set_xlim(0, 1); ax.set_xlabel("macro-F1"); ax.set_title("Model comparison (shared test set)")
for i, v in enumerate(summary["macro_F1"]): ax.text(v + .01, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 11. Predict on new text

In [ ]:
@torch.no_grad()
def predict(texts):
    texts = [texts] if isinstance(texts, str) else list(texts)
    enc = tokenizer(texts, truncation=True, max_length=128, padding=True, return_tensors="pt").to(model.device)
    probs = model(**enc).logits.softmax(-1).cpu().numpy()
    return [(LABELS[i], float(p[i])) for p, i in zip(probs, probs.argmax(1))]
demo = ["Not worth it.",
        "It works fine, nothing special.",
        "Absolutely love it, best purchase this year!"]
for t, (lab, conf) in zip(demo, predict(demo)):
    print(f"{lab:<9} {conf:.2f}  {t}")

# Part 2 — Product Category Clustering

## 2.1 Build a product-level table

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD
cat_raw = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/{f}", low_memory=False)[["name", "categories", "reviews.rating"]]
     for f in FILES], ignore_index=True)
cat_raw.columns = ["name", "categories", "rating"]
def clean_name(n):
    n = str(n).split("\n")[0]
    return re.sub(r",{2,}.*$", "", n).strip().strip(",").strip()
cat_raw["product"] = cat_raw["name"].map(clean_name)
cat_raw = cat_raw[(cat_raw["product"].str.lower() != "nan")
                  & (cat_raw["product"].str.len() > 0)
                  & cat_raw["categories"].notna()]
cat_raw["_catlen"] = cat_raw["categories"].astype(str).str.len()
products = (cat_raw.sort_values("_catlen", ascending=False)
                   .drop_duplicates("product")[["product", "categories"]]
                   .reset_index(drop=True))
print(f"{len(cat_raw):,} reviews  ->  {len(products)} unique products")
products.head(3)

## 2.2 Bag-of-words over the category tags

In [ ]:
def split_tags(s):
    return [t.strip().lower() for t in str(s).split(",") if t.strip()]
vec_cat = TfidfVectorizer(tokenizer=split_tags, preprocessor=lambda x: x,
                          token_pattern=None, min_df=2)
Xcat = vec_cat.fit_transform(products["categories"])
tags = np.array(vec_cat.get_feature_names_out())
print(f"product x tag matrix: {Xcat.shape}")

## 2.3 Choose the number of meta-categories

In [ ]:
sil = {k: silhouette_score(Xcat, KMeans(k, random_state=SEED, n_init=10).fit_predict(Xcat))
       for k in range(4, 7)}
plt.figure(figsize=(5, 3))
plt.plot(list(sil), list(sil.values()), "o-", color="#2b7bba")
plt.xticks(list(sil)); plt.xlabel("k (clusters)"); plt.ylabel("silhouette")
plt.title("Choosing k"); plt.tight_layout(); plt.show()
print("silhouette:", {k: round(v, 3) for k, v in sil.items()})
K = 5

## 2.4 Assign meta-categories (keyword rule)

In [ ]:
def assign_meta(name, cats=""):
    t = str(name).lower()
    if "fire tv" in t:
        return "Streaming & Other"
    if any(k in t for k in ["echo", "amazon tap", "cloud cam"]):
        return "Smart Speakers & Home"
    if any(k in t for k in ["paperwhite", "voyage", "oasis", "e-reader", "ereader", "ebook reader"]):
        return "E-readers"
    if any(k in t for k in ["tablet", "fire hd", "fire 7", "fire hdx", "kindle fire", "fire kids"]):
        return "Tablets"
    if any(k in t for k in ["batter", "charger", "charging", "cable", "adapter", "case", "cover",
                            "sleeve", "backpack", "stand", "mount", "power bank", "sd card",
                            "memory card", "flash drive", "hard drive", "protector"]):
        return "Accessories & Power"
    if "kindle" in t:
        return "E-readers"
    return "Streaming & Other"
products["meta_category"] = [assign_meta(p) for p in products["product"]]
META = [m for m in ["Tablets", "E-readers", "Smart Speakers & Home",
                    "Accessories & Power", "Streaming & Other"]
        if (products["meta_category"] == m).any()]
code = {m: k for k, m in enumerate(META)}
names = {k: m for m, k in code.items()}
K = len(META)
products["cluster"] = products["meta_category"].map(code)
glob = Xcat.mean(0).A1
class _Groups:
    labels_ = products["cluster"].to_numpy()
km = _Groups()
def distinctive(c, n=10):
    cen = Xcat[km.labels_ == c].mean(0).A1
    return tags[(cen - glob).argsort()[::-1][:n]]
print(products["meta_category"].value_counts())

## 2.5 What is in each cluster?

In [ ]:
rev_cat = cat_raw.merge(products[["product", "cluster", "meta_category"]], on="product")
summary = (rev_cat.groupby("meta_category")
           .agg(products=("product", "nunique"),
                reviews=("product", "size"),
                mean_rating=("rating", "mean"))
           .sort_values("reviews", ascending=False).round(2))
print(summary, "\n")
for c in range(K):
    prods = products.loc[products.cluster == c, "product"].tolist()
    print(f"\n[{names[c]}]  {len(prods)} products")
    print("   distinctive tags:", ", ".join(distinctive(c, 6)))
    print("   e.g.:", " | ".join(p[:40] for p in prods[:5]))

In [ ]:
_order = summary.index.tolist()
_pal = ["#2b7bba", "#e8912a", "#2ca02c", "#c0392b", "#9467bd", "#8c564b"]
_cmap = {nm: _pal[i % len(_pal)] for i, nm in enumerate(_order)}
_ncol = 3; _nrow = int(np.ceil(K / _ncol))
fig, axes = plt.subplots(_nrow, _ncol, figsize=(13, 3.2 * _nrow))
axes = np.array(axes).ravel()
for c in range(K):
    sc = Xcat[km.labels_ == c].mean(0).A1 - glob
    idx = sc.argsort()[::-1][:6][::-1]
    axes[c].barh(range(6), sc[idx], color=_cmap[names[c]])
    axes[c].set_yticks(range(6)); axes[c].set_yticklabels(tags[idx], fontsize=8)
    axes[c].set_title(names[c], fontsize=10); axes[c].set_xlabel("distinctiveness")
for j in range(K, len(axes)):
    axes[j].axis("off")
plt.suptitle("Top distinctive category tags per meta-category", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pal = ["#2b7bba", "#e8912a", "#2ca02c", "#c0392b", "#9467bd", "#8c564b"]
order = summary.index.tolist()
cmap = {nm: pal[i % len(pal)] for i, nm in enumerate(order)}
summary["reviews"].plot(kind="barh", ax=ax[0], color=[cmap[n] for n in summary.index])
ax[0].invert_yaxis(); ax[0].set_xlabel("reviews"); ax[0].set_title("Reviews per meta-category")
svd = TruncatedSVD(2, random_state=SEED).fit_transform(Xcat)
for c in range(K):
    m = km.labels_ == c
    ax[1].scatter(svd[m, 0], svd[m, 1], s=18, alpha=.7, color=cmap[names[c]], label=names[c])
ax[1].set_title("Products in 2-D (TF-IDF -> SVD)"); ax[1].legend(fontsize=8)
ax[1].set_xlabel("component 1"); ax[1].set_ylabel("component 2")
plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 3))
summary["mean_rating"].plot(kind="bar", color=[cmap[n] for n in summary.index])
plt.ylim(4, 5); plt.ylabel("mean stars"); plt.title("Average rating by meta-category")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

## 2.6 Attach the meta-category to every review and export

In [ ]:
out = rev_cat[["product", "meta_category", "rating"]].copy()
out_path = "artifacts/reviews_meta_categories.csv"
os.makedirs("artifacts", exist_ok=True)
out.to_csv(out_path, index=False)
print(f"wrote {len(out):,} rows -> {out_path}")
out.head()

# Part 3 — Review Summarization (Generative AI)

## 3.1 Setup

In [ ]:
import os, json, textwrap, getpass
import numpy as np, pandas as pd, matplotlib.pyplot as plt
MIN_REVIEWS = 20
OPENAI_MODEL = "gpt-4o-mini"
ARTICLE_DIR = "artifacts/articles"
os.makedirs(ARTICLE_DIR, exist_ok=True)

## 3.2 Review table: text + product + meta-category + sentiment

In [ ]:
sum_raw = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/{f}", low_memory=False)[["reviews.text", "name", "reviews.rating"]]
     for f in FILES], ignore_index=True)
sum_raw.columns = ["text", "name", "rating"]
sum_raw["text"] = sum_raw["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
sum_raw["rating"] = pd.to_numeric(sum_raw["rating"], errors="coerce")
sum_raw = sum_raw[(sum_raw["text"].str.len() >= 10) & sum_raw["rating"].between(1, 5)]
sum_raw["product"] = sum_raw["name"].map(clean_name)
sum_raw = sum_raw.drop_duplicates(subset=["text", "product"])
sum_raw = sum_raw.merge(products[["product", "meta_category"]], on="product", how="inner")
sum_raw["sentiment"] = pd.cut(sum_raw["rating"], [0, 2, 3, 5], labels=LABELS)
print(f"{len(sum_raw):,} reviews across {sum_raw['product'].nunique()} products / "
      f"{sum_raw['meta_category'].nunique()} categories")
sum_raw.head(3)

## 3.3 Aggregate per product

In [ ]:
def _share(s, lab): return float((s == lab).mean())
prod_stats = (sum_raw.groupby(["meta_category", "product"])
              .agg(n=("text", "size"), mean_rating=("rating", "mean"),
                   pos=("sentiment", lambda s: _share(s, "positive")),
                   neu=("sentiment", lambda s: _share(s, "neutral")),
                   neg=("sentiment", lambda s: _share(s, "negative")))
              .reset_index())
prod_stats = prod_stats[prod_stats["n"] >= MIN_REVIEWS].copy()
prod_stats["score"] = prod_stats["pos"] - prod_stats["neg"] + 0.02 * np.log1p(prod_stats["n"])
prod_stats = prod_stats.sort_values(["meta_category", "score"], ascending=[True, False])
print(f"{len(prod_stats)} products clear the >= {MIN_REVIEWS}-review bar")
prod_stats.head()

## 3.4 Complaint themes (KeyBERT, TF-IDF fallback)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import re
_GENERIC = set("""amazon kindle fire echo alexa tablet tablets device devices product products
one get got buy bought purchase purchased use used using would could really good great love
loved like nice best just also well item order ordered star stars thing things time really
easy little bit lot""".split())
def _name_tokens(product):
    return set(re.findall(r"[a-z]+", str(product).lower()))
def complaint_themes(neg_texts, pos_texts, product="", top_n=5, min_neg=8):
    """Terms over-represented in NEGATIVE vs POSITIVE reviews (log-odds w/ Dirichlet prior).
    Returns [] when there aren't enough negatives to say anything reliable."""
    neg = [t for t in neg_texts if isinstance(t, str) and len(t) > 15]
    pos = [t for t in pos_texts if isinstance(t, str) and len(t) > 15]
    if len(neg) < min_neg:
        return []
    block = _GENERIC | _name_tokens(product)
    vec = CountVectorizer(ngram_range=(1, 2), stop_words="english", min_df=2, max_features=4000)
    try:
        X = vec.fit_transform(neg + pos)
    except ValueError:
        return []
    terms = np.array(vec.get_feature_names_out())
    n = len(neg)
    neg_c = np.asarray(X[:n].sum(0)).ravel().astype(float)
    pos_c = np.asarray(X[n:].sum(0)).ravel().astype(float)
    a, V = 0.5, len(terms)
    score = (np.log((neg_c + a) / (neg_c.sum() + a * V))
             - np.log((pos_c + a) / (pos_c.sum() + a * V)))
    ok = np.where((neg_c >= 2) & (score > 0))[0]
    cand = sorted(((terms[i], score[i]) for i in ok
                   if not any(w in block for w in terms[i].split())),
                  key=lambda x: -x[1])
    return [t for t, _ in cand[:top_n]]
_p = prod_stats.iloc[0]["product"]
_sub = sum_raw[sum_raw["product"] == _p]
print("example complaints for:", _p[:50])
print(complaint_themes(_sub[_sub.sentiment == "negative"]["text"].tolist(),
                       _sub[_sub.sentiment == "positive"]["text"].tolist(), _p))

## 3.5 Build grounded facts per category

In [ ]:
def product_record(row):
    sub = sum_raw[sum_raw["product"] == row["product"]]
    negs = sub[sub.sentiment == "negative"]["text"].tolist()
    poss = sub[sub.sentiment == "positive"]["text"].tolist()
    return {"product": row["product"], "reviews": int(row["n"]),
            "mean_rating": round(float(row["mean_rating"]), 2),
            "pct_positive": round(row["pos"] * 100), "pct_negative": round(row["neg"] * 100),
            "top_complaints": complaint_themes(negs, poss, row["product"])}
category_facts = {}
for cat, grp in prod_stats.groupby("meta_category"):
    grp = grp.sort_values("score", ascending=False)
    top3 = [product_record(r) for _, r in grp.head(3).iterrows()]
    worst = None
    if len(grp) > 3:
        w = grp.iloc[-1]
        if w["neg"] >= 0.08 or w["mean_rating"] <= grp["mean_rating"].median() - 0.25:
            worst = product_record(w)
    category_facts[cat] = {"category": cat, "n_products": int(len(grp)),
                           "top3": top3, "worst": worst}
print(json.dumps(category_facts[list(category_facts)[0]], indent=2)[:1400])

## 3.6 Generate the articles (OpenAI)

In [ ]:
SYSTEM = ("You are a product-review editor. Write a concise, engaging recommendation article "
          "in Markdown from the JSON facts ONLY. Use ONLY the exact product names, numbers, ratings, "
          "and complaint phrases given in the JSON. Do NOT add any product features, specs, "
          "descriptions, or complaints that are not present in the data. "
          "Structure: a 1-2 sentence intro; a '## Top picks' section covering the top 3 (what sets each "
          "apart, using the ratings/sentiment and complaints given); and a '## One to skip' section for "
          "the worst product and why. If 'worst' is null, omit the 'One to skip' section entirely. ~200-300 words.")
def article_via_openai(facts):
    from openai import OpenAI
    client = OpenAI()
    r = client.chat.completions.create(
        model=OPENAI_MODEL, temperature=0,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": json.dumps(facts)}])
    return r.choices[0].message.content
def article_via_template(facts):
    L = [f"# {facts['category']} — what to buy", ""]
    L.append(f"*Based on customer reviews across {facts['n_products']} products in this category.*\n")
    L.append("## Top picks\n")
    for i, p in enumerate(facts["top3"], 1):
        comp = f" Common gripes: {', '.join(p['top_complaints'])}." if p["top_complaints"] else ""
        L.append(f"**{i}. {p['product']}** — {p['mean_rating']}/5 over {p['reviews']} reviews "
                 f"({p['pct_positive']}% positive).{comp}\n")
    if facts["worst"]:
        w = facts["worst"]
        comp = f" Reviewers flag: {', '.join(w['top_complaints'])}." if w["top_complaints"] else ""
        L.append("## One to skip\n")
        L.append(f"**{w['product']}** — only {w['mean_rating']}/5 over {w['reviews']} reviews "
                 f"({w['pct_negative']}% negative).{comp}")
    return "\n".join(L)
FORCE_TEMPLATE = False
USE_OPENAI = (not FORCE_TEMPLATE) and bool(os.environ.get("OPENAI_API_KEY", "").strip())
if not USE_OPENAI and not FORCE_TEMPLATE:
    try:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (blank = use template): ").strip()
        USE_OPENAI = bool(os.environ["OPENAI_API_KEY"])
    except Exception:
        USE_OPENAI = False
print("generation:", "OpenAI " + OPENAI_MODEL if USE_OPENAI else "template fallback")

In [ ]:
articles = {}
n_openai = n_fallback = 0
for cat, facts in category_facts.items():
    try:
        art = article_via_openai(facts) if USE_OPENAI else article_via_template(facts)
        if USE_OPENAI:
            n_openai += 1
        else:
            n_fallback += 1
    except Exception as e:
        print(f"OpenAI failed for {cat} ({type(e).__name__}: {e}) -> template")
        art = article_via_template(facts)
        n_fallback += 1
    articles[cat] = art
    fname = os.path.join(ARTICLE_DIR, cat.lower().replace(" & ", "_").replace(" ", "_") + ".md")
    open(fname, "w").write(art)
open(os.path.join(ARTICLE_DIR, "all_categories.md"), "w").write("\n\n---\n\n".join(articles.values()))
print(f"wrote {len(articles)} articles -> {ARTICLE_DIR}/  (OpenAI: {n_openai}, template: {n_fallback})")

## 3.7 Preview one article

In [ ]:
from IPython.display import Markdown, display
display(Markdown(articles[list(articles)[0]]))

## 3.8 Category insight graphs

In [ ]:
order = (sum_raw.groupby("meta_category").size().sort_values(ascending=False).index.tolist())
share = (sum_raw.groupby("meta_category")["sentiment"].value_counts(normalize=True)
         .unstack().reindex(order)[LABELS])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
share.plot(kind="barh", stacked=True, ax=ax[0],
           color=["#c0392b", "#e8912a", "#2b7bba"])
ax[0].invert_yaxis(); ax[0].set_xlabel("share of reviews"); ax[0].set_title("Sentiment mix by category")
ax[0].legend(title="", loc="lower right")
rows = []
for cat, f in category_facts.items():
    for p in f["top3"]:
        rows.append({"category": cat, "product": p["product"][:22], "rating": p["mean_rating"]})
top_df = pd.DataFrame(rows)
sns_order = top_df.groupby("category")["rating"].max().sort_values().index if False else order[::-1]
for i, cat in enumerate(order):
    sub = top_df[top_df.category == cat]
    ax[1].barh([f"{cat[:12]}: {p}" for p in sub["product"]], sub["rating"])
ax[1].set_xlim(3.5, 5); ax[1].set_xlabel("mean rating"); ax[1].set_title("Top-3 products per category")
plt.tight_layout(); plt.show()